# INCEpTION gold corpus - countable evaluation (E)

This notebook evaluates the pre-annotation review campaign using only countable,
structured corpus fields. It reads exclusively from `data/gcn_gold_corpus/` and does
not inspect, classify, or aggregate free-text comment content. Evidence and photometry
remain separate throughout.

## 1. Perimeter

The perimeter includes corpus rows, document records, and document-annotator
assignments. The expected 28 "annotators" are assignment rows; the corpus contains
fewer distinct people because one person may review multiple documents.

In [1]:
import importlib.util
import json
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 360)

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "data/gcn_gold_corpus").is_dir())
CORPUS = ROOT / "data/gcn_gold_corpus"
OUTPUT = ROOT / "data/interim/gcn_gold_eval"
OUTPUT.mkdir(parents=True, exist_ok=True)

COMMON_COLUMNS = [
    "document_name", "layer_source", "begin", "end", "span_index",
    "covered_text", "match_status", "changed_fields", "has_category",
    "is_annotator_note",
]
EVIDENCE_FEATURES = ["label", "target", "certainty", "value", "unit"]
PHOTOMETRY_FEATURES = [
    "measurement_type", "photometric_system", "target", "certainty",
    "magnitude_or_limit", "magnitude_error", "limit_sigma", "unit",
    "photometric_band", "obs_time_raw", "obs_time_type", "obs_time_reference",
    "exposure_time_raw", "timezone_raw", "instrument",
]
TABLES = {
    "evidence": pd.read_parquet(
        CORPUS / "evidence_spans.parquet",
        columns=COMMON_COLUMNS + EVIDENCE_FEATURES),
    "photometry": pd.read_parquet(
        CORPUS / "photometry_spans.parquet",
        columns=COMMON_COLUMNS + PHOTOMETRY_FEATURES),
}
assignments = pd.read_parquet(
    CORPUS / "annotators.parquet", columns=["document_name", "annotator"])
documents = pd.read_parquet(CORPUS / "documents.parquet")
assert all("comment" not in frame.columns for frame in TABLES.values())

expected = {
    "evidence_spans": 6615,
    "photometry_spans": 2741,
    "annotator_assignments": 28,
    "documents": 10,
}
observed = {
    "evidence_spans": len(TABLES["evidence"]),
    "photometry_spans": len(TABLES["photometry"]),
    "annotator_assignments": len(assignments),
    "documents": len(documents),
}
perimeter = pd.DataFrame([
    {"measure": name, "expected": expected[name], "observed": value,
     "status": "PASS" if value == expected[name] else "DEVIATION"}
    for name, value in observed.items()
])
display(perimeter)
print(f"Distinct annotator names: {assignments['annotator'].nunique()}")
print("Free-text comment columns loaded: 0")

,measure,expected,observed,status
0,evidence_spans,6615,6615,PASS
1,photometry_spans,2741,2741,PASS
2,annotator_assignments,28,28,PASS
3,documents,10,10,PASS


Distinct annotator names: 10
Free-text comment columns loaded: 0


## 2. Review depth

`n_baseline_reviewed` is `accepted + corrected + deleted`. The pair-level
`acceptance_rate` uses `accepted / (accepted + corrected)`, matching the first
acceptance definition in Part 3.

In [2]:
assignment_pairs = assignments[["document_name", "annotator"]].drop_duplicates()
pair_frames = []
for layer, frame in TABLES.items():
    reviewed = frame.loc[frame["layer_source"].ne("INITIAL_CAS")]
    counts = (reviewed.groupby(["document_name", "layer_source", "match_status"])
              .size().unstack(fill_value=0).reset_index()
              .rename(columns={"layer_source": "annotator"}))
    for status in ["accepted", "corrected", "created", "deleted"]:
        if status not in counts:
            counts[status] = 0
    profile = assignment_pairs.merge(
        counts, on=["document_name", "annotator"], how="left")
    profile[["accepted", "corrected", "created", "deleted"]] = profile[[
        "accepted", "corrected", "created", "deleted"]].fillna(0).astype(int)
    profile.insert(2, "layer", layer)
    profile = profile.rename(columns={
        "accepted": "n_accepted", "corrected": "n_corrected",
        "created": "n_created", "deleted": "n_deleted",
    })
    profile["n_baseline_reviewed"] = (
        profile["n_accepted"] + profile["n_corrected"] + profile["n_deleted"])
    denominator = profile["n_accepted"] + profile["n_corrected"]
    profile["acceptance_rate"] = profile["n_accepted"].div(
        denominator.where(denominator.ne(0)))
    pair_frames.append(profile[[
        "document_name", "annotator", "layer", "n_baseline_reviewed",
        "n_accepted", "n_corrected", "n_created", "n_deleted", "acceptance_rate"]])
per_pair_profile = pd.concat(pair_frames, ignore_index=True).sort_values(
    ["layer", "document_name", "annotator"]).reset_index(drop=True)
per_pair_profile.to_csv(
    OUTPUT / "per_pair_profile.csv", index=False, lineterminator="\n")
display(per_pair_profile)

,document_name,annotator,layer,n_baseline_reviewed,n_accepted,n_corrected,n_created,n_deleted,acceptance_rate
0,event_2025aji.xmi,Camille,evidence,201,188,9,3,4,0.954315
1,event_2025aji.xmi,Patrice,evidence,201,195,6,1,0,0.970149
2,event_2025aji.xmi,Xinyue,evidence,201,201,0,1,0,1.000000
3,event_2026owq.xmi,Dahlia,evidence,204,195,9,0,0,0.955882
4,event_2026owq.xmi,Sarah,evidence,204,197,7,5,0,0.965686
5,event_2026owq.xmi,Zhanat,evidence,204,196,8,1,0,0.960784
6,event_EP-260623_025405.xmi,Dahlia,evidence,96,91,5,10,0,0.947917
7,event_EP-260623_025405.xmi,Yodgor,evidence,96,96,0,0,0,1.000000
8,event_GCN-251013_173943.xmi,Andrii,evidence,237,237,0,0,0,1.000000
9,event_GCN-251013_173943.xmi,Eslam,evidence,237,237,0,0,0,1.000000


In [3]:
per_annotator_profile = (per_pair_profile.groupby(["annotator", "layer"], as_index=False)
    .agg(n_documents=("document_name", "nunique"),
         n_baseline_reviewed=("n_baseline_reviewed", "sum"),
         n_accepted=("n_accepted", "sum"), n_corrected=("n_corrected", "sum"),
         n_created=("n_created", "sum"), n_deleted=("n_deleted", "sum")))
denominator = per_annotator_profile["n_accepted"] + per_annotator_profile["n_corrected"]
per_annotator_profile["acceptance_rate"] = per_annotator_profile["n_accepted"].div(
    denominator.where(denominator.ne(0)))
per_annotator_profile = per_annotator_profile.sort_values(
    ["layer", "acceptance_rate", "annotator"]).reset_index(drop=True)
per_annotator_profile.to_csv(
    OUTPUT / "per_annotator_profile.csv", index=False, lineterminator="\n")
display(per_annotator_profile)

,annotator,layer,n_documents,n_baseline_reviewed,n_accepted,n_corrected,n_created,n_deleted,acceptance_rate
0,Zhanat,evidence,3,531,488,43,5,0,0.919021
1,Priyadarshini,evidence,3,574,530,44,40,0,0.923345
2,Camille,evidence,4,668,621,42,7,5,0.936652
3,Dahlia,evidence,3,559,524,35,18,0,0.937388
4,Sarah,evidence,3,587,555,32,21,0,0.945486
5,Patrice,evidence,3,342,327,15,2,0,0.956140
6,Eslam,evidence,3,462,449,13,15,0,0.971861
7,Andrii,evidence,1,237,237,0,0,0,1.000000
8,Xinyue,evidence,2,358,358,0,1,0,1.000000
9,Yodgor,evidence,3,501,501,0,4,0,1.000000


In [4]:
acceptance_ranges = []
for layer, rows in per_annotator_profile.groupby("layer"):
    minimum = rows["acceptance_rate"].min()
    maximum = rows["acceptance_rate"].max()
    acceptance_ranges.append({
        "layer": layer,
        "minimum_acceptance_rate": minimum,
        "minimum_annotators": ", ".join(sorted(
            rows.loc[rows["acceptance_rate"].eq(minimum), "annotator"])),
        "maximum_acceptance_rate": maximum,
        "maximum_annotators": ", ".join(sorted(
            rows.loc[rows["acceptance_rate"].eq(maximum), "annotator"])),
    })
acceptance_ranges = pd.DataFrame(acceptance_ranges)
display(acceptance_ranges)

,layer,minimum_acceptance_rate,minimum_annotators,maximum_acceptance_rate,maximum_annotators
0,evidence,0.919021,Zhanat,1.0,"Andrii, Xinyue, Yodgor"
1,photometry,0.070922,Dahlia,1.0,"Andrii, Xinyue"


Acceptance rates span **91.90%-100%** for evidence and **7.09%-100%** for
photometry. Therefore an acceptance rate from this campaign measures review depth and
editing practice as well as the pre-annotation system. The variation is reported
descriptively; it is not a judgement about the quality of any individual's work.

## 3. Acceptance rate: two countable definitions

A corrected row is **comment-only** exactly when parsed `changed_fields` equals
`["comment"]`. Every other corrected row contains at least one content field. Under
definition (b), a correction that only adds a comment does not modify the extracted data.

In [5]:
reviewed_frames = []
for layer, frame in TABLES.items():
    rows = frame.loc[frame["layer_source"].ne("INITIAL_CAS")].copy()
    rows["layer"] = layer
    rows["annotator"] = rows["layer_source"]
    rows["parsed_changed_fields"] = rows["changed_fields"].map(
        lambda value: json.loads(value) if isinstance(value, str) else [])
    reviewed_frames.append(rows)
reviewed_spans = pd.concat(reviewed_frames, ignore_index=True, sort=False)
corrected_rows = reviewed_spans.loc[reviewed_spans["match_status"].eq("corrected")].copy()
corrected_rows["correction_kind"] = np.where(
    corrected_rows["parsed_changed_fields"].map(lambda fields: fields == ["comment"]),
    "comment_only", "content")
correction_split = (corrected_rows.groupby(["layer", "correction_kind"]).size()
                    .unstack(fill_value=0).reset_index())
correction_split["total_corrected"] = (
    correction_split["comment_only"] + correction_split["content"])
correction_split.to_csv(
    OUTPUT / "correction_split.csv", index=False, lineterminator="\n")
display(correction_split)
print(f"Corpus-wide comment-only corrections: {int(correction_split['comment_only'].sum())}")
print(f"Corpus-wide content corrections: {int(correction_split['content'].sum())}")

correction_kind,layer,comment_only,content,total_corrected
0,evidence,167,57,224
1,photometry,276,375,651


Corpus-wide comment-only corrections: 443
Corpus-wide content corrections: 432


In [6]:
def acceptance_summary(rows, perimeter):
    output_rows = []
    for layer, layer_rows in rows.groupby("layer"):
        accepted = int(layer_rows["match_status"].eq("accepted").sum())
        corrected = int(layer_rows["match_status"].eq("corrected").sum())
        content_corrected = int((layer_rows["match_status"].eq("corrected") &
            layer_rows["parsed_changed_fields"].map(
                lambda fields: fields != ["comment"])).sum())
        output_rows.append({
            "perimeter": perimeter, "layer": layer, "n_accepted": accepted,
            "n_all_corrected": corrected, "n_content_corrected": content_corrected,
            "acceptance_all_corrections": accepted / (accepted + corrected),
            "acceptance_content_corrections_only": accepted / (
                accepted + content_corrected),
        })
    return pd.DataFrame(output_rows)


acceptance_rates = acceptance_summary(reviewed_spans, "all annotators")
active_annotators = {
    layer: set(rows.loc[rows["n_corrected"].gt(0), "annotator"])
    for layer, rows in per_annotator_profile.groupby("layer")
}
active_rows = pd.concat([
    reviewed_spans.loc[
        reviewed_spans["layer"].eq(layer) &
        reviewed_spans["annotator"].isin(names)]
    for layer, names in active_annotators.items()
], ignore_index=True)
acceptance_active = acceptance_summary(active_rows, "n_corrected > 0 in layer")
acceptance_active["excluded_annotators"] = acceptance_active["layer"].map(
    lambda layer: ", ".join(sorted(
        set(per_annotator_profile.loc[
            per_annotator_profile["layer"].eq(layer), "annotator"]) -
        active_annotators[layer])))
acceptance_rates.to_csv(
    OUTPUT / "acceptance_rates.csv", index=False, lineterminator="\n")
acceptance_active.to_csv(
    OUTPUT / "acceptance_rates_active_reviewers.csv", index=False, lineterminator="\n")
display(acceptance_rates)
display(acceptance_active)

,perimeter,layer,n_accepted,n_all_corrected,n_content_corrected,acceptance_all_corrections,acceptance_content_corrections_only
0,all annotators,evidence,4590,224,57,0.953469,0.987734
1,all annotators,photometry,1291,651,375,0.664779,0.774910


,perimeter,layer,n_accepted,n_all_corrected,n_content_corrected,acceptance_all_corrections,acceptance_content_corrections_only,excluded_annotators
0,n_corrected > 0 in layer,evidence,3494,224,57,0.939753,0.983948,"Andrii, Xinyue, Yodgor"
1,n_corrected > 0 in layer,photometry,871,651,375,0.572273,0.699037,"Andrii, Xinyue"


In [7]:
content_by_document = (corrected_rows.loc[corrected_rows["correction_kind"].eq("content")]
    .groupby(["document_name", "layer"]).size()
    .rename("n_content_corrected").reset_index())
acceptance_by_document = (per_pair_profile.groupby(
    ["document_name", "layer"], as_index=False)
    .agg(n_accepted=("n_accepted", "sum"), n_corrected=("n_corrected", "sum")))
acceptance_by_document = acceptance_by_document.merge(
    content_by_document, on=["document_name", "layer"], how="left")
acceptance_by_document["n_content_corrected"] = (
    acceptance_by_document["n_content_corrected"].fillna(0).astype(int))
acceptance_by_document["acceptance_all_corrections"] = (
    acceptance_by_document["n_accepted"] /
    (acceptance_by_document["n_accepted"] + acceptance_by_document["n_corrected"]))
acceptance_by_document["acceptance_content_corrections_only"] = (
    acceptance_by_document["n_accepted"] /
    (acceptance_by_document["n_accepted"] +
     acceptance_by_document["n_content_corrected"]))
acceptance_by_document = acceptance_by_document.sort_values(
    ["layer", "acceptance_all_corrections", "document_name"])
acceptance_by_document.to_csv(
    OUTPUT / "acceptance_by_document.csv", index=False, lineterminator="\n")
display(acceptance_by_document)

,document_name,layer,n_accepted,n_corrected,n_content_corrected,acceptance_all_corrections,acceptance_content_corrections_only
16,event_GRB-260708A.xmi,evidence,233,39,9,0.856618,0.962810
8,event_GCN-251222_170549.xmi,evidence,451,33,17,0.931818,0.963675
18,event_GRB241030.xmi,evidence,966,70,14,0.932432,0.985714
12,event_GCN-260614_134953.xmi,evidence,138,8,3,0.945205,0.978723
2,event_2026owq.xmi,evidence,588,24,1,0.960784,0.998302
10,event_GCN-260604_202037.xmi,evidence,282,10,1,0.965753,0.996466
4,event_EP-260623_025405.xmi,evidence,187,5,1,0.973958,0.994681
0,event_2025aji.xmi,evidence,584,15,8,0.974958,0.986486
6,event_GCN-251013_173943.xmi,evidence,696,15,0,0.978903,1.000000
14,event_GRB-241025_013651.xmi,evidence,465,5,3,0.989362,0.993590


## 4. Precision, recall, and F1

- **TP**: a baseline span kept by the annotator (`accepted` or `corrected`).
- **FP**: a baseline span deleted by the annotator.
- **FN**: a span created by the annotator with no baseline counterpart.

Created rows with `is_annotator_note=True` are excluded. Precision is an upper bound:
the annotation guide instructed reviewers to keep incorrect pre-annotations and explain
them in free text rather than delete them, so deletions are near zero. Part 4b presents
the bounded version using only structured rejection signals. Recall depends on review
depth because a missing span is counted only when a reviewer added it.

In [8]:
reviewed_spans["is_annotator_note"] = reviewed_spans["is_annotator_note"].fillna(False)
excluded_notes = reviewed_spans.loc[
    reviewed_spans["match_status"].eq("created") &
    reviewed_spans["is_annotator_note"]]
excluded_note_counts = (excluded_notes.groupby("layer").size()
                        .reindex(["evidence", "photometry"], fill_value=0)
                        .reset_index(name="excluded_annotator_notes"))
display(excluded_note_counts)


def metric_counts(rows):
    tp = int(rows["match_status"].isin(["accepted", "corrected"]).sum())
    fp = int(rows["match_status"].eq("deleted").sum())
    fn = int((rows["match_status"].eq("created") &
              ~rows["is_annotator_note"]).sum())
    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else np.nan
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision,
            "recall": recall, "f1": f1}


metrics_micro = pd.DataFrame([
    {"layer": layer, **metric_counts(rows)}
    for layer, rows in reviewed_spans.groupby("layer")
])
metrics_micro.to_csv(
    OUTPUT / "metrics_micro.csv", index=False, lineterminator="\n")
display(metrics_micro)

,layer,excluded_annotator_notes
0,evidence,11
1,photometry,36


,layer,tp,fp,fn,precision,recall,f1
0,evidence,4814,5,102,0.998962,0.979251,0.989009
1,photometry,1942,0,93,1.000000,0.954300,0.976616


In [9]:
annotator_metric_rows = []
for (annotator, layer), rows in reviewed_spans.groupby(["annotator", "layer"]):
    annotator_metric_rows.append({"annotator": annotator, "layer": layer,
                                  **metric_counts(rows)})
metrics_by_annotator = pd.DataFrame(annotator_metric_rows).merge(
    per_annotator_profile[["annotator", "layer", "n_corrected"]],
    on=["annotator", "layer"], validate="one_to_one")
metrics_by_annotator["zero_corrections"] = metrics_by_annotator["n_corrected"].eq(0)
metrics_by_annotator = metrics_by_annotator.sort_values(
    ["layer", "f1", "annotator"]).reset_index(drop=True)
metrics_by_annotator.to_csv(
    OUTPUT / "metrics_by_annotator.csv", index=False, lineterminator="\n")
display(metrics_by_annotator)

,annotator,layer,tp,fp,fn,precision,recall,f1,n_corrected,zero_corrections
0,Priyadarshini,evidence,574,0,40,1.000000,0.934853,0.966330,44,False
1,Eslam,evidence,462,0,15,1.000000,0.968553,0.984026,13,False
2,Dahlia,evidence,559,0,18,1.000000,0.968804,0.984155,35,False
3,Camille,evidence,663,5,7,0.992515,0.989552,0.991031,42,False
4,Sarah,evidence,587,0,10,1.000000,0.983250,0.991554,32,False
5,Zhanat,evidence,531,0,5,1.000000,0.990672,0.995314,43,False
6,Yodgor,evidence,501,0,4,1.000000,0.992079,0.996024,0,True
7,Patrice,evidence,342,0,2,1.000000,0.994186,0.997085,15,False
8,Xinyue,evidence,358,0,1,1.000000,0.997214,0.998605,0,True
9,Andrii,evidence,237,0,0,1.000000,1.000000,1.000000,0,True


In [10]:
category_metric_rows = []
for layer, rows in reviewed_spans.groupby("layer"):
    category_column = "label" if layer == "evidence" else "measurement_type"
    item_key = ["document_name", "begin", "end", "span_index"]
    baseline = TABLES[layer].loc[
        TABLES[layer]["layer_source"].eq("INITIAL_CAS"),
        item_key + [category_column]].rename(
            columns={category_column: "baseline_category"})
    categorized = rows.merge(baseline, on=item_key, how="left", validate="many_to_one")
    categorized["evaluation_category"] = np.where(
        categorized["match_status"].eq("created"),
        categorized[category_column], categorized["baseline_category"])
    categorized["evaluation_category"] = categorized[
        "evaluation_category"].fillna("<UNSET>")
    for category, category_rows in categorized.groupby("evaluation_category"):
        counts = metric_counts(category_rows)
        support = counts["tp"] + counts["fp"] + counts["fn"]
        category_metric_rows.append({
            "layer": layer, "category": category, "support": support,
            "interpretable": support >= 20, **counts,
        })
metrics_by_category = pd.DataFrame(category_metric_rows).sort_values(
    ["layer", "f1", "category"], na_position="first").reset_index(drop=True)
metrics_by_category.to_csv(
    OUTPUT / "metrics_by_category.csv", index=False, lineterminator="\n")
display(metrics_by_category)

,layer,category,support,interpretable,tp,fp,fn,precision,recall,f1
0,evidence,<UNSET>,3,False,0,0,3,NaN,0.000000,NaN
1,evidence,LOCALIZATION,257,True,221,2,34,0.991031,0.866667,0.924686
2,evidence,TRIGGER_INSTRUMENT,433,True,411,0,22,1.000000,0.949192,0.973934
3,evidence,REDSHIFT_CONTEXT,20,True,19,0,1,1.000000,0.950000,0.974359
4,evidence,REDSHIFT_EVENT,145,True,139,0,6,1.000000,0.958621,0.978873
5,evidence,TRIGGER_TIME,219,True,212,2,5,0.990654,0.976959,0.983759
6,evidence,COUNTERPART_ASSOCIATION,509,True,495,0,14,1.000000,0.972495,0.986056
7,evidence,SPECTROSCOPY,116,True,113,0,3,1.000000,0.974138,0.986900
8,evidence,HIGH_ENERGY_PROPERTY,439,True,430,0,9,1.000000,0.979499,0.989643
9,evidence,CLASSIFICATION_INTERPRETATION,108,True,107,0,1,1.000000,0.990741,0.995349


## 4b. Declared rejections: a lower bound on precision

### 4b.1 Signal A: certainty changed to `rejected`

Signal A requires an exact matched pair with `match_status == "corrected"`,
`"certainty"` in the parsed `changed_fields`, a baseline certainty other than
`rejected`, and an annotator certainty equal to `rejected`. The test reads only
structured span columns; no free-text comment is loaded or classified.


In [11]:
ITEM_KEY = ["document_name", "begin", "end", "span_index"]
declared_pair_frames = []
for layer, frame in TABLES.items():
    category_column = "label" if layer == "evidence" else "measurement_type"
    baseline = frame.loc[
        frame["layer_source"].eq("INITIAL_CAS"),
        ITEM_KEY + [category_column, "certainty", "covered_text"],
    ].rename(columns={
        category_column: "baseline_category",
        "certainty": "baseline_certainty",
        "covered_text": "baseline_text",
    })
    matched = reviewed_spans.loc[
        reviewed_spans["layer"].eq(layer)
        & reviewed_spans["match_status"].isin(["accepted", "corrected"])
    ].copy()
    matched = matched.rename(columns={category_column: "annotator_category"})
    paired = matched.merge(baseline, on=ITEM_KEY, how="left", validate="many_to_one")
    paired["signal_a"] = (
        paired["match_status"].eq("corrected")
        & paired["parsed_changed_fields"].map(lambda fields: "certainty" in fields)
        & paired["baseline_certainty"].fillna("").ne("rejected")
        & paired["certainty"].fillna("").eq("rejected")
    )
    annotator_blank = (paired["annotator_category"].isna()
                       | paired["annotator_category"].astype("string").str.strip().eq(""))
    baseline_present = (paired["baseline_category"].notna()
                        & paired["baseline_category"].astype("string").str.strip().ne(""))
    paired["signal_b"] = (baseline_present & ~paired["has_category"].fillna(False)
                           & annotator_blank)
    declared_pair_frames.append(paired)

declared_pairs = pd.concat(declared_pair_frames, ignore_index=True, sort=False)
signal_a_rows = declared_pairs.loc[declared_pairs["signal_a"]].copy()
signal_b_rows = declared_pairs.loc[declared_pairs["signal_b"]].copy()
declared_rows = declared_pairs.loc[
    declared_pairs[["signal_a", "signal_b"]].any(axis=1)].copy()
declared_rows["signals"] = declared_rows.apply(
    lambda row: "+".join(name for name, hit in
                         [("A", row["signal_a"]), ("B", row["signal_b"])] if hit), axis=1)
declared_export = declared_rows[[
    "signals", "layer", "document_name", "annotator", "begin", "end",
    "span_index", "baseline_category", "annotator_category", "baseline_text",
    "baseline_certainty", "certainty", "match_status", "has_category",
]].sort_values(["layer", "document_name", "annotator", "begin", "end"])
declared_export.to_csv(
    OUTPUT / "declared_rejections.csv", index=False, lineterminator="\n")

signal_a_by_layer = (signal_a_rows.groupby("layer").size()
                     .reindex(["evidence", "photometry"], fill_value=0)
                     .reset_index(name="signal_a"))
signal_a_by_annotator = (signal_a_rows.groupby(["layer", "annotator"]).size()
                         .reset_index(name="signal_a"))
signal_a_by_category = (signal_a_rows.groupby(["layer", "baseline_category"]).size()
                        .reset_index(name="signal_a"))
signal_a_listing = signal_a_rows[[
    "document_name", "annotator", "layer", "baseline_category", "baseline_text",
    "baseline_certainty", "certainty",
]].rename(columns={"baseline_category": "label_or_measurement_type",
                   "baseline_text": "span_text",
                   "certainty": "annotator_certainty"})
display(signal_a_by_layer)
display(signal_a_by_annotator)
display(signal_a_by_category)
display(signal_a_listing)
print(f"Declared rejection rows exported: {len(declared_export)}")


,layer,signal_a
0,evidence,14
1,photometry,0


,layer,annotator,signal_a
0,evidence,Camille,14


,layer,baseline_category,signal_a
0,evidence,LOCALIZATION,1
1,evidence,TRIGGER_INSTRUMENT,11
2,evidence,TRIGGER_TIME,2


,document_name,annotator,layer,label_or_measurement_type,span_text,baseline_certainty,annotator_certainty
2125,event_GCN-251222_170549.xmi,Camille,evidence,TRIGGER_INSTRUMENT,SVOM/ECLAIRs,confirmed,rejected
2128,event_GCN-251222_170549.xmi,Camille,evidence,TRIGGER_INSTRUMENT,SVOM/GRM,confirmed,rejected
2132,event_GCN-251222_170549.xmi,Camille,evidence,TRIGGER_INSTRUMENT,SVOM/ECLAIRs,confirmed,rejected
2133,event_GCN-251222_170549.xmi,Camille,evidence,TRIGGER_TIME,2025-12-22T17:05:46 UTC,confirmed,rejected
2142,event_GCN-251222_170549.xmi,Camille,evidence,TRIGGER_INSTRUMENT,SVOM/ECLAIRs,confirmed,rejected
2280,event_GCN-251222_170549.xmi,Camille,evidence,TRIGGER_INSTRUMENT,SVOM ECLAIRs,confirmed,rejected
2281,event_GCN-251222_170549.xmi,Camille,evidence,TRIGGER_INSTRUMENT,GRM,confirmed,rejected
2330,event_GCN-251222_170549.xmi,Camille,evidence,TRIGGER_INSTRUMENT,SVOM/ECLAIRs,confirmed,rejected
3043,event_GRB-241025_013651.xmi,Camille,evidence,LOCALIZATION,uncertainty of 4.8 arcsec,confirmed,rejected
3514,event_GRB-260708A.xmi,Camille,evidence,TRIGGER_INSTRUMENT,Fermi-LAT,confirmed,rejected


Declared rejection rows exported: 20


### 4b.2 Signal B: category cleared

Signal B is an `accepted` or `corrected` matched pair whose baseline category is
non-null and non-blank, while the annotator row has both
`has_category.fillna(False) == False` and a null or blank category feature. The category
feature is `label` for evidence and `measurement_type` for photometry.


In [12]:
signal_b_by_layer = (signal_b_rows.groupby("layer").size()
                     .reindex(["evidence", "photometry"], fill_value=0)
                     .reset_index(name="signal_b"))
signal_b_by_annotator = (signal_b_rows.groupby(["layer", "annotator"]).size()
                         .reset_index(name="signal_b"))
signal_b_listing = signal_b_rows[[
    "document_name", "annotator", "layer", "baseline_category", "baseline_text",
    "annotator_category", "has_category", "match_status",
]].rename(columns={"baseline_category": "label_or_measurement_type",
                   "baseline_text": "span_text"})
signal_overlap = declared_pairs.loc[
    declared_pairs["signal_a"] & declared_pairs["signal_b"]]
display(signal_b_by_layer)
display(signal_b_by_annotator)
display(signal_b_listing)
print(f"Signal A/B overlap: {len(signal_overlap)}")
print(f"Deduplicated declared rejections: {len(declared_rows)}")


,layer,signal_b
0,evidence,4
1,photometry,2


,layer,annotator,signal_b
0,evidence,Camille,3
1,evidence,Sarah,1
2,photometry,Camille,1
3,photometry,Sarah,1


,document_name,annotator,layer,label_or_measurement_type,span_text,annotator_category,has_category,match_status
2116,event_GCN-251222_170549.xmi,Camille,evidence,TRIGGER_TIME,17:05:51 UT on 22 Dec 2025,None,False,corrected
2324,event_GCN-251222_170549.xmi,Camille,evidence,REDSHIFT_EVENT,redshift of z = 3.171,None,False,corrected
2703,event_GCN-260604_202037.xmi,Sarah,evidence,REDSHIFT_EVENT,z = 0.86 +/- 0.09,None,False,corrected
3143,event_GRB-241025_013651.xmi,Camille,evidence,COUNTERPART_ASSOCIATION,optical afterglow,None,False,corrected
4894,event_2025aji.xmi,Camille,photometry,detection,37.92 19.32 0.11 i,None,False,corrected
5958,event_GCN-251013_173943.xmi,Sarah,photometry,upper_limit,"13347 | 2025-10-13 21:21:09 | MASTER-SAAO | (23h 03m 02.87s , -00d 29m 15.8s) | C | 120 | 19.9 |",None,False,corrected


Signal A/B overlap: 0
Deduplicated declared rejections: 20


### 4b.3 Precision as a bounded interval

The upper bound retains Part 4's definition (`FP = deleted`). For the lower bound,
deduplicated Signal A and Signal B rows move from TP to FP:
`FP = deleted + declared rejection` and `TP = kept - declared rejection`.


In [13]:
def precision_bounds(rows, declared_count):
    upper = metric_counts(rows)
    lower_tp = upper["tp"] - declared_count
    lower_fp = upper["fp"] + declared_count
    lower_precision = lower_tp / (lower_tp + lower_fp) if lower_tp + lower_fp else np.nan
    return {
        "kept_upper": upper["tp"], "deleted": upper["fp"],
        "declared_rejections": declared_count,
        "precision_lower": lower_precision, "precision_upper": upper["precision"],
    }

layer_bounds = []
for layer, rows in reviewed_spans.groupby("layer"):
    n_declared = int(declared_rows["layer"].eq(layer).sum())
    layer_bounds.append({"layer": layer, **precision_bounds(rows, n_declared)})
precision_bounds_by_layer = pd.DataFrame(layer_bounds)

annotator_bounds = []
for (annotator, layer), rows in reviewed_spans.groupby(["annotator", "layer"]):
    n_declared = int((declared_rows["annotator"].eq(annotator)
                      & declared_rows["layer"].eq(layer)).sum())
    annotator_bounds.append({
        "annotator": annotator, "layer": layer,
        **precision_bounds(rows, n_declared),
    })
precision_bounds_by_annotator = pd.DataFrame(annotator_bounds).sort_values(
    ["layer", "annotator"]).reset_index(drop=True)
precision_bounds_by_layer.to_csv(
    OUTPUT / "precision_bounds.csv", index=False, lineterminator="\n")
precision_bounds_by_annotator.to_csv(
    OUTPUT / "precision_bounds_by_annotator.csv", index=False, lineterminator="\n")
display(precision_bounds_by_layer)
display(precision_bounds_by_annotator)


,layer,kept_upper,deleted,declared_rejections,precision_lower,precision_upper
0,evidence,4814,5,18,0.995227,0.998962
1,photometry,1942,0,2,0.998970,1.000000


,annotator,layer,kept_upper,deleted,declared_rejections,precision_lower,precision_upper
0,Andrii,evidence,237,0,0,1.000000,1.000000
1,Camille,evidence,663,5,17,0.967066,0.992515
2,Dahlia,evidence,559,0,0,1.000000,1.000000
3,Eslam,evidence,462,0,0,1.000000,1.000000
4,Patrice,evidence,342,0,0,1.000000,1.000000
5,Priyadarshini,evidence,574,0,0,1.000000,1.000000
6,Sarah,evidence,587,0,1,0.998296,1.000000
7,Xinyue,evidence,358,0,0,1.000000,1.000000
8,Yodgor,evidence,501,0,0,1.000000,1.000000
9,Zhanat,evidence,531,0,0,1.000000,1.000000


### 4b.4 Coverage of the structured signal

Coverage is counted across all assigned annotator names. Signal A and Signal B are
practices observed in the data, not campaign-wide rejection instructions.


In [14]:
all_annotators = set(assignments["annotator"].dropna().unique())
signal_a_annotators = set(signal_a_rows["annotator"])
signal_b_annotators = set(signal_b_rows["annotator"])
either_annotators = signal_a_annotators | signal_b_annotators
neither_annotators = all_annotators - either_annotators
signal_coverage = pd.DataFrame([
    {"coverage": "all assigned", "n_annotators": len(all_annotators),
     "annotators": ", ".join(sorted(all_annotators))},
    {"coverage": "Signal A", "n_annotators": len(signal_a_annotators),
     "annotators": ", ".join(sorted(signal_a_annotators))},
    {"coverage": "Signal B", "n_annotators": len(signal_b_annotators),
     "annotators": ", ".join(sorted(signal_b_annotators))},
    {"coverage": "either signal", "n_annotators": len(either_annotators),
     "annotators": ", ".join(sorted(either_annotators))},
    {"coverage": "neither signal", "n_annotators": len(neither_annotators),
     "annotators": ", ".join(sorted(neither_annotators))},
])
layer_signal_coverage = pd.DataFrame([
    {"layer": layer,
     "signal_a_annotators": signal_a_rows.loc[
         signal_a_rows["layer"].eq(layer), "annotator"].nunique(),
     "signal_b_annotators": signal_b_rows.loc[
         signal_b_rows["layer"].eq(layer), "annotator"].nunique()}
    for layer in ["evidence", "photometry"]
])
display(signal_coverage)
display(layer_signal_coverage)


,coverage,n_annotators,annotators
0,all assigned,10,"Andrii, Camille, Dahlia, Eslam, Patrice, Priyadarshini, Sarah, Xinyue, Yodgor, Zhanat"
1,Signal A,1,Camille
2,Signal B,2,"Camille, Sarah"
3,either signal,2,"Camille, Sarah"
4,neither signal,8,"Andrii, Dahlia, Eslam, Patrice, Priyadarshini, Xinyue, Yodgor, Zhanat"


,layer,signal_a_annotators,signal_b_annotators
0,evidence,1,2
1,photometry,0,2


### 4b.5 Finding

The campaign had no structured field for recording that a pre-annotation should not
exist. Rejections were therefore left either in non-aggregable free-text comments or
improvised through existing schema fields such as `certainty`. Precision can only be
bounded, not measured: even the lower bound is incomplete because the guide did not
document `certainty=rejected` as a rejection mechanism and described photometry
certainty only as generally `confirmed`. The concrete requirement for a future campaign
is a dedicated rejection field, either a boolean or an explicitly reserved value,
rather than a free-text convention.


## 5. Where the extractor fails

Exact span pairing and structured feature changes are measured independently. Only
corrected rows containing at least one non-comment field enter the field analysis.

In [15]:
pairing_rows = []
for layer, rows in reviewed_spans.groupby("layer"):
    matched = int(rows["match_status"].isin(["accepted", "corrected"]).sum())
    deleted = int(rows["match_status"].eq("deleted").sum())
    pairing_rows.append({
        "layer": layer, "matched_baseline_spans": matched,
        "deleted_baseline_spans": deleted,
        "exact_baseline_span_pairing_rate": matched / (matched + deleted),
    })
pairing_rates = pd.DataFrame(pairing_rows)
pairing_rates.to_csv(
    OUTPUT / "pairing_rates.csv", index=False, lineterminator="\n")
display(pairing_rates)

,layer,matched_baseline_spans,deleted_baseline_spans,exact_baseline_span_pairing_rate
0,evidence,4814,5,0.998962
1,photometry,1942,0,1.000000


In [16]:
field_rows = []
matched_counts = (reviewed_spans.loc[
    reviewed_spans["match_status"].isin(["accepted", "corrected"])]
    .groupby("layer").size())
for layer, rows in corrected_rows.loc[
        corrected_rows["correction_kind"].eq("content")].groupby("layer"):
    field_counts = pd.Series([
        field
        for fields in rows["parsed_changed_fields"]
        for field in sorted(set(fields))
    ]).value_counts()
    for field, count in field_counts.items():
        field_rows.append({
            "layer": layer, "field": field, "rows_changed": int(count),
            "matched_pairs": int(matched_counts[layer]),
            "percent_of_matched_pairs": 100 * count / matched_counts[layer],
        })
field_change_rates = pd.DataFrame(field_rows).sort_values(
    ["layer", "rows_changed", "field"], ascending=[True, False, True])
field_change_rates.to_csv(
    OUTPUT / "field_change_rates.csv", index=False, lineterminator="\n")
display(field_change_rates)

,layer,field,rows_changed,matched_pairs,percent_of_matched_pairs
0,evidence,comment,44,4814,0.914001
1,evidence,certainty,24,4814,0.498546
2,evidence,value,15,4814,0.311591
4,evidence,label,9,4814,0.186955
3,evidence,target,9,4814,0.186955
5,evidence,unit,3,4814,0.062318
6,photometry,comment,298,1942,15.345005
7,photometry,instrument,213,1942,10.968074
8,photometry,photometric_system,131,1942,6.745623
9,photometry,obs_time_reference,124,1942,6.385170


Exact baseline-span pairing is **99.90%** for evidence and **100%** for photometry.
Among evidence content corrections, `certainty` differs on 24 of 4,814 matched pairs
(**0.50%**), `value` on 15 (**0.31%**), and `label` and `target` on 9 each
(**0.19%**). Photometry attributes change much more often: `instrument` on 213 of
1,942 pairs (**10.97%**), `photometric_system` on 131 (**6.75%**), and
`obs_time_reference` on 124 (**6.39%**). Thus the extractor places spans almost
perfectly but fills some photometry fields less reliably. Comment changes that
accompany content corrections remain counted, but their text is not interpreted.

## 6. Inter-annotator agreement

One item is one baseline span identified by
`(document_name, begin, end, span_index)`. Its value is the annotator's final nominal
value for the feature. Deletions are the explicit category `<DELETED>`; null and empty
final values remain visible as `<NULL>` and `<EMPTY>`.

The `krippendorff` package is not installed in the project environment, so nominal
alpha is implemented directly below using Krippendorff's coincidence matrix.

In [17]:
print(f"krippendorff package installed: {importlib.util.find_spec('krippendorff') is not None}")


def final_nominal_value(row, feature):
    if row.match_status == "deleted":
        return "<DELETED>"
    value = getattr(row, feature)
    if pd.isna(value):
        return "<NULL>"
    if isinstance(value, str) and value == "":
        return "<EMPTY>"
    return str(value)


def nominal_alpha(ratings):
    groups = [group["value"].tolist()
              for _, group in ratings.groupby("item", sort=False)
              if len(group) >= 2]
    if not groups:
        return {"alpha": np.nan, "raw_agreement": np.nan, "n_items": 0,
                "n_annotators": ratings["annotator"].nunique(),
                "agreeing_pairs": 0, "disagreeing_pairs": 0,
                "category_distribution": "{}",
                "degenerate_reason": "no item has 2 ratings"}
    categories = sorted({value for values in groups for value in values})
    category_index = {value: index for index, value in enumerate(categories)}
    coincidence = np.zeros((len(categories), len(categories)), dtype=float)
    agreeing_pairs = 0
    all_pairs = 0
    for values in groups:
        n_values = len(values)
        counts = pd.Series(values).value_counts()
        all_pairs += n_values * (n_values - 1) // 2
        agreeing_pairs += sum(int(count * (count - 1) // 2) for count in counts)
        for first, n_first in counts.items():
            for second, n_second in counts.items():
                coincidence[category_index[first], category_index[second]] += (
                    n_first * (n_second - (first == second)) / (n_values - 1))
    total = coincidence.sum()
    observed_disagreement = (total - np.trace(coincidence)) / total
    marginals = coincidence.sum(axis=1)
    expected_disagreement = (
        (total * total - np.square(marginals).sum()) / (total * (total - 1))
        if total > 1 else np.nan)
    if expected_disagreement == 0:
        alpha = np.nan
        reason = "no expected disagreement: all usable ratings have one category"
    else:
        alpha = 1 - observed_disagreement / expected_disagreement
        reason = ""
    distribution = pd.Series(
        [value for values in groups for value in values]
    ).value_counts().sort_index().to_dict()
    return {
        "alpha": alpha, "raw_agreement": agreeing_pairs / all_pairs,
        "n_items": len(groups), "n_annotators": ratings["annotator"].nunique(),
        "agreeing_pairs": agreeing_pairs,
        "disagreeing_pairs": all_pairs - agreeing_pairs,
        "category_distribution": json.dumps(distribution, sort_keys=True),
        "degenerate_reason": reason,
    }


AGREEMENT_FEATURES = {
    "evidence": ["label", "certainty"],
    "photometry": ["measurement_type", "instrument", "photometric_system",
                   "obs_time_reference"],
}
alpha_rows = []
for layer, features in AGREEMENT_FEATURES.items():
    layer_rows = reviewed_spans.loc[
        reviewed_spans["layer"].eq(layer) &
        reviewed_spans["match_status"].isin(["accepted", "corrected", "deleted"])
    ].copy()
    layer_rows["item"] = (
        layer_rows["document_name"] + "|" + layer_rows["begin"].astype(str) + "|" +
        layer_rows["end"].astype(str) + "|" + layer_rows["span_index"].astype(str))
    active_names = active_annotators[layer]
    for feature in features:
        ratings = layer_rows.copy()
        ratings["value"] = [final_nominal_value(row, feature)
                            for row in ratings.itertuples()]
        perimeters = {
            "full": ratings,
            "restricted": ratings.loc[ratings["annotator"].isin(active_names)],
        }
        documents_for_layer = sorted(ratings["document_name"].unique())
        for perimeter_name, perimeter_rows in perimeters.items():
            for document_name in documents_for_layer + ["POOLED"]:
                selected = (perimeter_rows if document_name == "POOLED" else
                            perimeter_rows.loc[
                                perimeter_rows["document_name"].eq(document_name)])
                alpha_rows.append({
                    "perimeter": perimeter_name, "layer": layer,
                    "feature": feature, "document_name": document_name,
                    **nominal_alpha(selected[["item", "annotator", "value"]]),
                })
alpha_results = pd.DataFrame(alpha_rows)
alpha_results.to_csv(
    OUTPUT / "krippendorff_alpha.csv", index=False, lineterminator="\n")
print("Annotators retained by the restricted perimeter:")
for layer, names in active_annotators.items():
    excluded = sorted(set(per_annotator_profile.loc[
        per_annotator_profile["layer"].eq(layer), "annotator"]) - names)
    print(f"{layer}: retained={len(names)}; excluded={', '.join(excluded)}")
display(alpha_results.loc[alpha_results["perimeter"].eq("full")])
display(alpha_results.loc[alpha_results["perimeter"].eq("restricted")])

krippendorff package installed: False


Annotators retained by the restricted perimeter:
evidence: retained=7; excluded=Andrii, Xinyue, Yodgor
photometry: retained=8; excluded=Andrii, Xinyue


,perimeter,layer,feature,document_name,alpha,raw_agreement,n_items,n_annotators,agreeing_pairs,disagreeing_pairs,category_distribution,degenerate_reason
0,full,evidence,label,event_2025aji.xmi,0.981968,0.986733,201,3,595,8,"{""<DELETED>"": 4, ""CLASSIFICATION_INTERPRETATION"": 3, ""COUNTERPART_ASSOCIATION"": 75, ""EVENT_IDENTITY"": 284, ""HIGH_ENERGY_PROPERTY"": 36, ""LIGHTCURVE_EVOLUTION"": 81, ""LOCALIZATION"": 25, ""NEGATIVE_STATEMENT"": 3, ""REDSHIFT_CONTEXT"": 9, ""REDSHIFT_EVENT"": 15, ""SPECTROSCOPY"": 24, ""T90"": 3, ""TRIGGER_INSTRUMENT"": 21, ""TRIGGER_TIME"": 20}",
1,full,evidence,label,event_2026owq.xmi,1.000000,1.000000,204,3,612,0,"{""CLASSIFICATION_INTERPRETATION"": 21, ""COUNTERPART_ASSOCIATION"": 66, ""EVENT_IDENTITY"": 309, ""HIGH_ENERGY_PROPERTY"": 42, ""HOST_CONTEXT"": 3, ""LIGHTCURVE_EVOLUTION"": 75, ""LOCALIZATION"": 15, ""REDSHIFT_CONTEXT"": 3, ""REDSHIFT_EVENT"": 21, ""SPECTROSCOPY"": 24, ""T90"": 6, ""TRIGGER_INSTRUMENT"": 15, ""TRIGGER_TIME"": 12}",
2,full,evidence,label,event_EP-260623_025405.xmi,1.000000,1.000000,96,2,96,0,"{""CLASSIFICATION_INTERPRETATION"": 2, ""COUNTERPART_ASSOCIATION"": 24, ""DURATION_GENERAL"": 4, ""EVENT_IDENTITY"": 82, ""HIGH_ENERGY_PROPERTY"": 16, ""LIGHTCURVE_EVOLUTION"": 18, ""LOCALIZATION"": 12, ""REDSHIFT_EVENT"": 14, ""SPECTROSCOPY"": 4, ""TRIGGER_INSTRUMENT"": 8, ""TRIGGER_TIME"": 8}",
3,full,evidence,label,event_GCN-251013_173943.xmi,1.000000,1.000000,237,3,711,0,"{""CLASSIFICATION_INTERPRETATION"": 18, ""COUNTERPART_ASSOCIATION"": 90, ""DURATION_GENERAL"": 3, ""EVENT_IDENTITY"": 333, ""HIGH_ENERGY_PROPERTY"": 33, ""LIGHTCURVE_EVOLUTION"": 69, ""LOCALIZATION"": 30, ""REDSHIFT_EVENT"": 27, ""SPECTROSCOPY"": 9, ""T90"": 12, ""TRIGGER_INSTRUMENT"": 54, ""TRIGGER_TIME"": 33}",
4,full,evidence,label,event_GCN-251222_170549.xmi,0.979171,0.983471,242,2,238,4,"{""<NULL>"": 2, ""CLASSIFICATION_INTERPRETATION"": 6, ""COUNTERPART_ASSOCIATION"": 48, ""DURATION_GENERAL"": 4, ""EVENT_IDENTITY"": 192, ""HIGH_ENERGY_PROPERTY"": 56, ""LIGHTCURVE_EVOLUTION"": 36, ""LOCALIZATION"": 22, ""NEGATIVE_STATEMENT"": 2, ""REDSHIFT_CONTEXT"": 2, ""REDSHIFT_EVENT"": 13, ""SPECTROSCOPY"": 8, ""T90"": 6, ""TRIGGER_INSTRUMENT"": 60, ""TRIGGER_TIME"": 27}",
5,full,evidence,label,event_GCN-260604_202037.xmi,0.991032,0.993151,146,2,145,1,"{""<NULL>"": 1, ""CLASSIFICATION_INTERPRETATION"": 10, ""COUNTERPART_ASSOCIATION"": 18, ""DURATION_GENERAL"": 4, ""EVENT_IDENTITY"": 128, ""HIGH_ENERGY_PROPERTY"": 44, ""HOST_CONTEXT"": 2, ""LIGHTCURVE_EVOLUTION"": 18, ""LOCALIZATION"": 12, ""REDSHIFT_EVENT"": 5, ""T90"": 8, ""TRIGGER_INSTRUMENT"": 30, ""TRIGGER_TIME"": 12}",
6,full,evidence,label,event_GCN-260614_134953.xmi,1.000000,1.000000,73,2,73,0,"{""CLASSIFICATION_INTERPRETATION"": 6, ""COUNTERPART_ASSOCIATION"": 4, ""EVENT_IDENTITY"": 40, ""HIGH_ENERGY_PROPERTY"": 16, ""LIGHTCURVE_EVOLUTION"": 2, ""LOCALIZATION"": 18, ""T90"": 10, ""TRIGGER_INSTRUMENT"": 34, ""TRIGGER_TIME"": 16}",
7,full,evidence,label,event_GRB-241025_013651.xmi,0.988700,0.991507,157,3,467,4,"{""<DELETED>"": 1, ""<NULL>"": 1, ""CLASSIFICATION_INTERPRETATION"": 9, ""COUNTERPART_ASSOCIATION"": 29, ""DURATION_GENERAL"": 6, ""EVENT_IDENTITY"": 216, ""HIGH_ENERGY_PROPERTY"": 63, ""LIGHTCURVE_EVOLUTION"": 15, ""LOCALIZATION"": 27, ""REDSHIFT_CONTEXT"": 3, ""REDSHIFT_EVENT"": 12, ""SPECTROSCOPY"": 12, ""T90"": 9, ""TRIGGER_INSTRUMENT"": 45, ""TRIGGER_TIME"": 23}",
8,full,evidence,label,event_GRB-260708A.xmi,0.982117,0.985294,68,4,402,6,"{""CLASSIFICATION_INTERPRETATION"": 12, ""COUNTERPART_ASSOCIATION"": 28, ""DURATION_GENERAL"": 4, ""EVENT_IDENTITY"": 96, ""HIGH_ENERGY_PROPERTY"": 26, ""LIGHTCURVE_EVOLUTION"": 18, ""LOCALIZATION"": 16, ""SPECTROSCOPY"": 4, ""T90"": 8, ""TRIGGER_INSTRUMENT"": 36, ""TRIGGER_TIME"": 24}",
9,full,evidence,label,event_GRB241030.xmi,0.997428,0.998069,259,4,1551,3,"{""CLASSIFICATION_INTERPRETATION"": 20, ""COUNTERPART_ASSOCIATION"": 112, ""DURATION_GENERAL"": 8, ""EVENT_IDENTITY"": 476, ""HIGH_ENERGY_PROPERTY"": 100, ""H

,perimeter,layer,feature,document_name,alpha,raw_agreement,n_items,n_annotators,agreeing_pairs,disagreeing_pairs,category_distribution,degenerate_reason
11,restricted,evidence,label,event_2025aji.xmi,0.973008,0.980100,201,2,197,4,"{""<DELETED>"": 4, ""CLASSIFICATION_INTERPRETATION"": 2, ""COUNTERPART_ASSOCIATION"": 50, ""EVENT_IDENTITY"": 189, ""HIGH_ENERGY_PROPERTY"": 24, ""LIGHTCURVE_EVOLUTION"": 54, ""LOCALIZATION"": 16, ""NEGATIVE_STATEMENT"": 2, ""REDSHIFT_CONTEXT"": 6, ""REDSHIFT_EVENT"": 10, ""SPECTROSCOPY"": 16, ""T90"": 2, ""TRIGGER_INSTRUMENT"": 14, ""TRIGGER_TIME"": 13}",
12,restricted,evidence,label,event_2026owq.xmi,1.000000,1.000000,204,3,612,0,"{""CLASSIFICATION_INTERPRETATION"": 21, ""COUNTERPART_ASSOCIATION"": 66, ""EVENT_IDENTITY"": 309, ""HIGH_ENERGY_PROPERTY"": 42, ""HOST_CONTEXT"": 3, ""LIGHTCURVE_EVOLUTION"": 75, ""LOCALIZATION"": 15, ""REDSHIFT_CONTEXT"": 3, ""REDSHIFT_EVENT"": 21, ""SPECTROSCOPY"": 24, ""T90"": 6, ""TRIGGER_INSTRUMENT"": 15, ""TRIGGER_TIME"": 12}",
13,restricted,evidence,label,event_EP-260623_025405.xmi,NaN,NaN,0,1,0,0,{},no item has 2 ratings
14,restricted,evidence,label,event_GCN-251013_173943.xmi,1.000000,1.000000,237,2,237,0,"{""CLASSIFICATION_INTERPRETATION"": 12, ""COUNTERPART_ASSOCIATION"": 60, ""DURATION_GENERAL"": 2, ""EVENT_IDENTITY"": 222, ""HIGH_ENERGY_PROPERTY"": 22, ""LIGHTCURVE_EVOLUTION"": 46, ""LOCALIZATION"": 20, ""REDSHIFT_EVENT"": 18, ""SPECTROSCOPY"": 6, ""T90"": 8, ""TRIGGER_INSTRUMENT"": 36, ""TRIGGER_TIME"": 22}",
15,restricted,evidence,label,event_GCN-251222_170549.xmi,0.979171,0.983471,242,2,238,4,"{""<NULL>"": 2, ""CLASSIFICATION_INTERPRETATION"": 6, ""COUNTERPART_ASSOCIATION"": 48, ""DURATION_GENERAL"": 4, ""EVENT_IDENTITY"": 192, ""HIGH_ENERGY_PROPERTY"": 56, ""LIGHTCURVE_EVOLUTION"": 36, ""LOCALIZATION"": 22, ""NEGATIVE_STATEMENT"": 2, ""REDSHIFT_CONTEXT"": 2, ""REDSHIFT_EVENT"": 13, ""SPECTROSCOPY"": 8, ""T90"": 6, ""TRIGGER_INSTRUMENT"": 60, ""TRIGGER_TIME"": 27}",
16,restricted,evidence,label,event_GCN-260604_202037.xmi,NaN,NaN,0,1,0,0,{},no item has 2 ratings
17,restricted,evidence,label,event_GCN-260614_134953.xmi,1.000000,1.000000,73,2,73,0,"{""CLASSIFICATION_INTERPRETATION"": 6, ""COUNTERPART_ASSOCIATION"": 4, ""EVENT_IDENTITY"": 40, ""HIGH_ENERGY_PROPERTY"": 16, ""LIGHTCURVE_EVOLUTION"": 2, ""LOCALIZATION"": 18, ""T90"": 10, ""TRIGGER_INSTRUMENT"": 34, ""TRIGGER_TIME"": 16}",
18,restricted,evidence,label,event_GRB-241025_013651.xmi,0.983073,0.987261,157,2,155,2,"{""<DELETED>"": 1, ""<NULL>"": 1, ""CLASSIFICATION_INTERPRETATION"": 6, ""COUNTERPART_ASSOCIATION"": 19, ""DURATION_GENERAL"": 4, ""EVENT_IDENTITY"": 144, ""HIGH_ENERGY_PROPERTY"": 42, ""LIGHTCURVE_EVOLUTION"": 10, ""LOCALIZATION"": 18, ""REDSHIFT_CONTEXT"": 2, ""REDSHIFT_EVENT"": 8, ""SPECTROSCOPY"": 8, ""T90"": 6, ""TRIGGER_INSTRUMENT"": 30, ""TRIGGER_TIME"": 15}",
19,restricted,evidence,label,event_GRB-260708A.xmi,0.982117,0.985294,68,4,402,6,"{""CLASSIFICATION_INTERPRETATION"": 12, ""COUNTERPART_ASSOCIATION"": 28, ""DURATION_GENERAL"": 4, ""EVENT_IDENTITY"": 96, ""HIGH_ENERGY_PROPERTY"": 26, ""LIGHTCURVE_EVOLUTION"": 18, ""LOCALIZATION"": 16, ""SPECTROSCOPY"": 4, ""T90"": 8, ""TRIGGER_INSTRUMENT"": 36, ""TRIGGER_TIME"": 24}",
20,restricted,evidence,label,event_GRB241030.xmi,0.996572,0.997426,259,3,775,2,"{""CLASSIFICATION_INTERPRETATION"": 15, ""COUNTERPART_ASSOCIATION"": 84, ""DURATION_GENERAL"": 6, ""EVENT_IDENTITY"": 357, ""HIGH_ENERGY_PROPERTY"": 75, ""HOST_CONTEXT"": 9, ""LIGHTCURVE_EVOLUTION"": 33, ""LOCALIZATION"": 33, ""NEGATIVE_STATEMENT"": 3, ""REDSHIFT_CONTEXT"": 2, ""REDSHIFT_EVENT"": 22, ""SPECTROSCOPY"": 21, ""T90"": 9, ""TRIGGER_INSTRUMENT"": 81, ""TRIGGER_TIME"": 27}",


These are not conventional reliability figures. Reviewers edited a shared
pre-annotation rather than annotating independently from scratch, so agreement partly
measures that nobody changed the supplied value. Full/restricted pooled alpha and raw
agreement are **0.9924/0.9902** and **99.47%/99.34%** for evidence `label`, and
**0.9939/0.9920** and **99.79%/99.82%** for photometry `measurement_type`. In
contrast, photometry `instrument` is **0.7070/0.6413** with **80.94%/70.49%** raw
agreement, `photometric_system` is **0.8318/0.8117** with **89.75%/83.59%**, and
`obs_time_reference` is **0.8120/0.7623** with **87.83%/79.10%**. The scheme is
substantially more stable about what the span represents than about these attributes.

## 7. Declared limitations

- Comments were free text with no controlled vocabulary, so their content is not
  aggregated. The countable record shows that 443 corrections modify only `comment`.
- Precision's upper bound counts only deletions as false positives. Part 4b reports a
  schema-only lower bound alongside it.
- The campaign had no structured rejection mechanism. The lower bound is itself a floor
  because it captures only annotators who improvised rejection through existing fields.
- Recall depends on review depth because an omission appears only when a reviewer creates
  the missing span.
- There is no curated consensus layer; the corpus contains independent validations of a
  shared pre-annotation.
- Two annotators produced no corrections in either layer. Their unchanged rows provide no
  observed correction signal.